In [1]:
import mlflow
import pandas as pd
import os
from workspace.sources.experiments.metrics import Precision, standard_evaluation_metrics, standard_metrics, Loss
from workspace.sources.experiments.visualizations.plots import plot_confusion_matrix, plot_roc_curve

from workspace.sources.experiments.visualizations.tables import METRICS_PLOT_NAMES_MAPPING
from workspace.notebooks.utils import flatten_dict, get_run_artifacts_path, get_experiment_best_runs_by_metrics
import ast

mlflow.set_tracking_uri('../../mlruns')

In [2]:
experiment_name = 'bert'

### ReCOVery Dataset

In [3]:
dataset_name = 'recovery'

best_runs = get_experiment_best_runs_by_metrics(experiment_name, dataset_name)

#### Confusion Matrix


In [6]:
def produce_experiment_confusion_matrices(model_best_runs_by_metrics, dataset_name):
    for metric in standard_evaluation_metrics:
        best_run = model_best_runs_by_metrics[metric.name]
        best_model_run_id = best_run.info.run_id
        run_artifacts_path = get_run_artifacts_path(best_model_run_id)
        plot_confusion_matrix(run_artifacts_path,
                              metric,
                              dataset_name=dataset_name,
                              show_plot=False)


produce_experiment_confusion_matrices(best_runs, dataset_name)

#### ROC Curve

In [7]:
def produce_experiment_roc_curves(model_best_runs_by_metrics, dataset_name):
    for metric in standard_evaluation_metrics:
        best_run = model_best_runs_by_metrics[metric.name]
        best_model_run_id = best_run.info.run_id
        run_artifacts_path = get_run_artifacts_path(best_model_run_id)
        plot_roc_curve(run_artifacts_path,
                       metric,
                       dataset_name=dataset_name,
                       show_plot=False)


produce_experiment_roc_curves(best_runs, dataset_name)

#### Hyperparameters Tables

In [14]:
def hparams_by_metric_to_latex_table(hparams_by_metric, dataset_name, columns):
    for metric, model_hparams in hparams_by_metric.items():
        df = pd.DataFrame(list(model_hparams.items()), columns=columns)
        output_dir = f'assets/{dataset_name}/model_hyperparameters/'
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, f'model_hyperparameters_table_by_{metric.name}.tex')
        float_format_fn = lambda x: f'{x:.3f}' if isinstance(x, (float, int)) else str(x).replace('_', '\\_')
        latex_table = df.to_latex(index=False,
                                  escape=True,
                                  float_format=float_format_fn,
                                  column_format='|c||c|')
        # Add horizontal lines between each row
        latex_table = latex_table.replace('\\\\', '\\\\ \\midrule')
        with open(output_path, 'w') as f:
            f.write(latex_table)


def produce_experiment_model_hparams_tables(model_best_runs_by_metrics, dataset_name):
    hyperparameters_by_metric = dict()
    for metric in standard_evaluation_metrics:
        model_hparams_dict_str = model_best_runs_by_metrics[metric.name].data.params['model_input_hyperparameters']
        model_hparams_dict = ast.literal_eval(model_hparams_dict_str)

        hyperparameters_by_metric[metric] = model_hparams_dict

    hparams_by_metric_to_latex_table(hyperparameters_by_metric, dataset_name, columns=['Hyperparameter', 'Value'])


produce_experiment_model_hparams_tables(best_runs, dataset_name)

##### Preprocessing Pipeline

In [ ]:
def produce_experiment_pipeline_hparams_tables(model_best_runs_by_metrics, dataset_name):
    hyperparameters_by_metric = dict()
    for metric in standard_evaluation_metrics:
        pipeline_name = best_runs[metric.name].data.params['preprocessing_pipeline_name']
        hyperparameters_by_metric['pipeline_name'] = pipeline_name

        pipeline = ast.literal_eval(best_runs[metric.name].data.params['preprocessing_pipeline'])
        hyperparameters_by_metric
        for step, preprocessing in enumerate(pipeline):
            preprocessing_params = ast.literal_eval(best_runs[metric.name].data.params[preprocessing])
            for param, value in preprocessing_params.items():
                

            hyperparameters_by_metric[f'step_{step}'] = preprocessing_table_value
        model_hparams_dict_str = model_best_runs_by_metrics[metric.name].data.params['model_input_hyperparameters']
        model_hparams_dict = ast.literal_eval(model_hparams_dict_str)

        hyperparameters_by_metric[metric] = model_hparams_dict

    hparams_by_metric_to_latex_table(hyperparameters_by_metric, dataset_name, columns=['Hyperparameter', 'Value'])

In [10]:
pipeline = ast.literal_eval(best_runs['precision'].data.params['preprocessing_pipeline'])
pipeline

['bert_base_uncased_encoder']

In [12]:
pipeline[0]

'bert_base_uncased_encoder'

In [13]:
best_runs['precision'].data.params[pipeline[0]]

"{'tokenizer': 'bert-base-uncased', 'truncation': True, 'padding': 'max_length', 'truncation_max_length': 256, 'is_split_into_words': False}"